## Imports

In [ ]:
import os, warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import healpy as hp
from scipy.optimize import curve_fit
from scipy.stats import chi2
import astropy.units as u
from astropy.coordinates import SkyCoord, EarthLocation
from astropy.table import Table
import utils

pd.set_option("display.max_columns", None)
warnings.filterwarnings("ignore")

observing_location = EarthLocation.of_site("Roque de los Muchachos")


## Configuration

Edit this cell to customise the run.

In [ ]:
ROOT       = Path(os.getcwd())
ROOT_DATA  = ROOT / "data"
DCHECK_DIR = ROOT_DATA / "datachecks"
OUT_DIR    = ROOT_DATA / "selection"
OUT_DIR.mkdir(parents=True, exist_ok=True)

FNAME_TABLE_SRUN  = DCHECK_DIR / "table_datachecks_srunwise.csv"
FNAME_DCHECK_FLAT = DCHECK_DIR / "datachecks_flat.parquet"
FNAME_SELECTED_RUNS = OUT_DIR / "selected_runs.csv"
FNAME_RAW_LY3   = OUT_DIR / "raw_srw_ly3.ecsv"     # input: more accurate srun-wise LY values
FNAME_FINAL_LY3 = OUT_DIR / "final_srw_ly3.ecsv"   # output: linearly-interpolated srun-wise LY

SOURCE_NAME     = "Crab"
WOBBLE_DISTANCE = 0.4 * u.deg
POINTING_UNCERT = 0.1 * u.deg

DATE_MIN = datetime.fromisoformat("2025-08-17")   # None to disable
DATE_MAX = None

DUST_MAX         = None    # µg/m³
ZD_MAX           = 90      # deg
NSB_MAX          = None
ELAPSED_TIME_MIN = 120.0   # s
ELAPSED_TIME_MAX = 2100.0  # s
DRDI_FIT_ERROR   = False

THR_FIT = 3e-3

## Step 1 - Load subrun-wise table

In [ ]:
df = pd.read_csv(FNAME_TABLE_SRUN)
df["timestamp"] = pd.to_datetime(df["timestamp"])
print(f"Loaded {len(df):,} runs  ({df['timestamp'].min().date()} → {df['timestamp'].max().date()})")
df.head(3)


## Step 2 - Dataset overview

In [ ]:
plot_cfg = [
    ("ZD_corrected_cosmics_rate_at_422_pe", "ZD-corr. cosmics rate [Hz]", 40),
    ("diffuse_nsb_std",                     "Diffuse NSB std",            40),
    ("zd",                                  "Mean ZD [deg]",              40),
    ("tng_dust",                            "TNG Dust [µg/m³]",           40),
    ("humidity",                            "Humidity [%]",               40),
    ("telapsed",                            "Elapsed time [s]",           40),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
for ax, (col, label, bins) in zip(axes.ravel(), plot_cfg):
    if col in df.columns:
        ax.hist(df[col].dropna(), bins=bins, color="lightgray", edgecolor="none")
        ax.set(xlabel=label, ylabel="Runs", yscale="log")
        ax.grid(alpha=0.3)
    else:
        ax.text(0.5, 0.5, f"{col!r}\nnot in table", ha="center", va="center", transform=ax.transAxes)
        ax.axis("off")
plt.tight_layout(); plt.show()


## Step 3 - Date filter

In [ ]:
df_sel = df.copy()
for attr, op in [("DATE_MIN", ">="), ("DATE_MAX", "<=")]:
    val = globals()[attr]
    if val is not None:
        n_before = len(df_sel)
        df_sel = df_sel[df_sel["timestamp"].apply(lambda t: eval(f"t {op} pd.Timestamp(val)"))]
        print(f"  {attr} ({val.date()}): -{n_before - len(df_sel):,} → {len(df_sel):,} remaining")
    else:
        print(f"  {attr}: no cut")
print(f"After date filter: {len(df_sel):,} runs")


## Step 4 - Quality cuts

In [ ]:
cuts = [
    ("tng_dust",      "<=", DUST_MAX,         "Dust ≤ {v} µg/m³"),
    ("diffuse_nsb_std","<=", NSB_MAX,          "NSB_std ≤ {v}"),
    ("zd",            "<=", ZD_MAX,           "ZD ≤ {v}°"),
]

for col, op, val, label in cuts:
    if val is not None and col in df_sel.columns:
        n = len(df_sel)
        df_sel = df_sel[df_sel[col].isna() | df_sel[col].apply(lambda x: eval(f"x {op} val"))]
        print(f"  {label.format(v=val)}: -{n - len(df_sel):,} → {len(df_sel):,}")
    else:
        print(f"  {label.format(v=val)}: disabled")

# Elapsed time
if "telapsed" in df_sel.columns:
    n = len(df_sel)
    mask = pd.Series(True, index=df_sel.index)
    if ELAPSED_TIME_MIN: mask &= df_sel["telapsed"] >= ELAPSED_TIME_MIN
    if ELAPSED_TIME_MAX: mask &= df_sel["telapsed"] <= ELAPSED_TIME_MAX
    df_sel = df_sel[mask]
    print(f"  Elapsed time [{ELAPSED_TIME_MIN/60:.0f}–{ELAPSED_TIME_MAX/60:.0f} min]: -{n - len(df_sel):,} → {len(df_sel):,}")

# DRDI flag
if DRDI_FIT_ERROR and "run_drdi_fit_error_flag" in df_sel.columns:
    n = len(df_sel)
    df_sel = df_sel[~df_sel["run_drdi_fit_error_flag"].astype(bool)]
    print(f"  DRDI fit error: -{n - len(df_sel):,} → {len(df_sel):,}")

print(f"\nAfter quality cuts: {len(df_sel):,} runs ({len(df_sel)/len(df)*100:.1f}% of dataset)")


## Step 5 - Wobble-ring matching

In [ ]:
source_coord = SkyCoord.from_name(SOURCE_NAME)
run_coords   = SkyCoord(ra=df_sel["ra"].to_numpy() * u.deg, dec=df_sel["dec"].to_numpy() * u.deg)
separation   = source_coord.separation(run_coords)

min_dist = WOBBLE_DISTANCE - POINTING_UNCERT
max_dist = WOBBLE_DISTANCE + POINTING_UNCERT
df_sel["separation_deg"] = separation.deg
df_sel = df_sel[(separation >= min_dist) & (separation <= max_dist)].copy()

print(f"Wobble ring [{min_dist:.2f}, {max_dist:.2f}] around {SOURCE_NAME}")
print(f"  RA={source_coord.ra.deg:.3f}°  Dec={source_coord.dec.deg:.3f}°")
print(f"  Matching runs: {len(df_sel):,}  |  Total time: {df_sel['telapsed'].sum()/3600:.2f} h")


## Step 6 - Pointing diagnostics

In [ ]:
# AltAz polar map
theta  = np.radians(df_sel["az"].to_numpy() % 360)
r      = df_sel["zd"].to_numpy()
counts, t_edges, r_edges = np.histogram2d(theta, r, bins=[60, 30], range=[[0, 2*np.pi], [0, 90]])
T, R   = np.meshgrid(t_edges, r_edges)

fig, axes = plt.subplots(1, 2, figsize=(10, 5), gridspec_kw={"width_ratios": [1, 1]})

ax_p = fig.add_subplot(121, projection="polar")
axes[0].remove()
pc = ax_p.pcolormesh(T, R, counts.T, cmap="viridis")
plt.colorbar(pc, ax=ax_p, orientation="horizontal", pad=0.12, label="Counts")
ax_p.set(theta_zero_location="N", theta_direction=-1, rmax=90, title="Alt–Az")

# Wobble ring scatter
ax2 = axes[1]
azimuths = np.linspace(0, 360, 300) * u.deg
circle     = source_coord.directional_offset_by(azimuths, WOBBLE_DISTANCE)
circle_in  = source_coord.directional_offset_by(azimuths, WOBBLE_DISTANCE - POINTING_UNCERT)
circle_out = source_coord.directional_offset_by(azimuths, WOBBLE_DISTANCE + POINTING_UNCERT)
sc = ax2.scatter(df_sel["ra"], df_sel["dec"], s=8, alpha=0.6, c=df_sel["zd"], cmap="plasma")
ax2.plot(source_coord.ra.deg, source_coord.dec.deg, "*r", label=SOURCE_NAME)
ax2.plot(circle.ra.deg, circle.dec.deg, "r--", lw=1.5, label=f"Wobble ({WOBBLE_DISTANCE.value:.2f}°)")
ax2.plot(circle_in.ra.deg, circle_in.dec.deg, "r:", lw=0.8)
ax2.plot(circle_out.ra.deg, circle_out.dec.deg, "r:", lw=0.8, label=f"±{POINTING_UNCERT.value:.2f}° tol.")
plt.colorbar(sc, ax=ax2, label="Mean ZD [deg]")
ax2.set(xlabel="RA [deg]", ylabel="Dec [deg]", title=f"{len(df_sel)} selected runs - {SOURCE_NAME}")
ax2.legend(fontsize=8); ax2.grid(alpha=0.3, ls=":")

plt.suptitle(f"Pointing - {SOURCE_NAME} wobble ring")
plt.tight_layout(); plt.show()


## Step 7 - Timeline & dust inspection

In [ ]:
ts = df_sel["timestamp"]
fig, axes = plt.subplots(4, 1, figsize=(8, 8), sharex=True)

axes[0].scatter(ts, df_sel["ZD_corrected_cosmics_rate_at_422_pe"], s=6, alpha=0.5, color="royalblue")
axes[0].set(ylabel="ZD-corr. rate [Hz]"); axes[0].grid(alpha=0.3)

if "tng_dust" in df_sel.columns:
    axes[1].scatter(ts, df_sel["tng_dust"], s=6, alpha=0.5, color="saddlebrown")
    if DUST_MAX: axes[1].axhline(DUST_MAX, color="red", lw=1, ls="--", label=f"DUST_MAX={DUST_MAX}")
    axes[1].set(ylabel="TNG Dust [µg/m³]", yscale="log"); axes[1].grid(alpha=0.3)

if "light_yield" in df_sel.columns:
    axes[2].scatter(ts, df_sel["light_yield"], s=6, alpha=0.5, color="darkorange")
    axes[2].set(ylabel="Light Yield", ylim=(0.5, 1.2)); axes[2].grid(alpha=0.3)

axes[3].scatter(ts, df_sel["telapsed"] / 60, s=6, alpha=0.5, color="seagreen")
if ELAPSED_TIME_MIN: axes[3].axhline(ELAPSED_TIME_MIN / 60, color="red", lw=1, ls="--")
axes[3].set(ylabel="Elapsed [min]", xlabel="Date"); axes[3].grid(alpha=0.3)
axes[3].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%b"))
axes[3].xaxis.set_major_locator(mdates.MonthLocator(interval=2))

fig.suptitle(f"Selected runs - {SOURCE_NAME}")
plt.gcf().autofmt_xdate(rotation=0, ha="center"); plt.tight_layout(); plt.show()

# Dust vs light yield
converted_dates = mdates.date2num(ts)
fig, ax = plt.subplots(figsize=(5, 3.5))
sc = ax.scatter(df_sel["tng_dust"], df_sel["light_yield"], marker=".", s=df_sel["telapsed"]**0.7, c=converted_dates)
cbar = fig.colorbar(sc, ax=ax)
cbar.ax.yaxis.set_major_formatter(mdates.DateFormatter("%Y-%b-%d"))
ax.set(xlabel="TNG dust [µg/m³]", ylabel="Light yield", ylim=(0.6, 1.1), xlim=(5e-3, 30), xscale="log")
ax.grid(alpha=0.3); plt.show()


## Brightest triplet - per-run LY fits

In [ ]:
raw_ly3 = pd.read_csv(FNAME_RAW_LY3)
# raw_ly3 = pd.read_csv(FNAME_RAW_LY3, comment='#', sep="\s+")

raw_ly3 = raw_ly3.rename(columns={"subrun": "srun"})

raw_ly3_runs = set(raw_ly3["runnumber"].astype(int))
raw_ly3_lookup = {
    (int(r), int(s)): float(ly)
    for r, s, ly in zip(raw_ly3["runnumber"], raw_ly3["srun"], raw_ly3["light_yield"])
}

print(f"Loaded {len(raw_ly3):,} srun-wise LY rows from {len(raw_ly3_runs):,} runs -> {FNAME_RAW_LY3}")
raw_ly3.head()

## Discarded runs

In [ ]:
discarded_runs = [
    21727, 21948, 21949, 21950, 21951, 23091, 23010, 23895, 22986, 22569, 21947,
    21405, 21921, 21939, 22327, 22598, 22826, 22825, 23671, 23562, 
    23491
]

In [ ]:
df_flat = pd.read_parquet(FNAME_DCHECK_FLAT) if FNAME_DCHECK_FLAT.exists() else None
rw_run, rw_time, rw_flag, rw_median, rw_std, rw_dust, rw_zd, rw_intensity = [], [], [], [], [], [], [], []
fit_results = {}  # run_id -> dict(p0, p1, fit_ok, subrun, tstamp) for ALL subruns of the run

for obs_id, row in df_sel.iterrows():
    run_id = int(row["obs_id"])
    if df_flat is None:
        print(f"Run {run_id}: flat parquet missing - skip"); continue

    if run_id not in raw_ly3_runs:
        print(f"Run {run_id}: not in {FNAME_RAW_LY3.name} - skip"); continue

    tab = df_flat[df_flat["runnumber"] == run_id].copy()
    if tab.empty:
        print(f"Run {run_id}: no data - skip"); continue

    pd_time = pd.to_datetime(tab["time"])
    tstamp  = np.array([t.timestamp() for t in pd_time]) / 1e9
    dtime   = pd_time.to_numpy()
    subrun  = tab["subrun"].to_numpy()
    dust    = float(row["tng_dust"]) if "tng_dust" in row and not pd.isna(row["tng_dust"]) else np.nan

    # Use the more accurate srun-wise LY values from raw_srw_ly3.ecsv instead
    # of tab["light_yield"]; subruns missing from that file become NaN and are
    # dropped by the mask below, just like any other missing/outlier point.
    ly = np.array([raw_ly3_lookup.get((run_id, int(s)), np.nan) for s in subrun])

    # Mask: drop NaNs + 2-sigma outliers
    mean_s, std_s = np.nanmean(ly), np.nanstd(ly)
    mask = ~(np.isnan(tstamp) | np.isnan(ly) | (ly > 2) |
             (ly > mean_s + 2*std_s) | (ly < mean_s - 2*std_s))
    x_fit, y_fit = tstamp[mask], ly[mask]

    fit_ok = False; p0 = p1 = chi2_ndf = pvalue = np.nan
    if len(x_fit) >= 2:
        try:
            params, _, info, _, _ = curve_fit(utils.straight_line, x_fit, y_fit,
                                              p0=[np.nanmean(y_fit), 0], full_output=True)
            p0, p1 = params
            c2 = float(np.sum(info["fvec"]**2)); ndf = len(x_fit) - 2
            chi2_ndf = c2 / ndf if ndf > 0 else np.nan
            pvalue   = chi2.sf(c2, ndf) if ndf > 0 else np.nan
            fit_ok   = (c2 / len(x_fit) < THR_FIT) and not np.isnan(p0)
        except Exception as e:
            print(f"  Run {run_id}: fit failed - {e}")

    # Store the per-run fit + ALL subrun timestamps so we can later predict
    # light_yield for every subrun of runs in the final selection (Step 8).
    fit_results[run_id] = dict(p0=p0, p1=p1, fit_ok=fit_ok, subrun=subrun, tstamp=tstamp)

    y_median = np.nanmedian(ly)
    xarr_ts  = np.linspace(np.nanmin(tstamp), np.nanmax(tstamp), 200) if len(tstamp) else np.array([])
    xarr_dt  = np.array([datetime.fromtimestamp(t * 1e9) for t in xarr_ts])

    fig, ax = plt.subplots(figsize=(7, 3))
    ax.errorbar(dtime, ly, ls="-", marker="", color="k", zorder=-10, label="Data")
    ax.errorbar(dtime[mask], ly[mask], ls="", marker="")  # highlight fitted points

    if fit_ok:
        ax.plot(xarr_dt, utils.straight_line(xarr_ts, p0, p1), "g--",
                label=f"Fit OK  χ²/ndf={chi2_ndf:.0e}")
    else:
        label = f"Fit not OK  χ²/ndf={chi2_ndf:.1e}" if not np.isnan(chi2_ndf) else "Fit not OK"
        if not np.isnan(p0):
            ax.plot(xarr_dt, utils.straight_line(xarr_ts, p0, p1), "r--", label=label)
        ax.axhline(y_median, color="darkorange", ls=":", label=f"Median={y_median:.3f}")

    ax.legend(frameon=False, loc=(1.03, 0))
    if not np.isnan(dust):
        cdust = "r" if dust > 3 else "g" if dust < 1.5 else "darkorange"
        ax.text(0.03, 1.05, f"TNG dust: {dust:.2f} µg/m³", ha="left", va="center",
                transform=ax.transAxes, color=cdust, fontsize=9)
    if not np.isnan(pvalue):
        ax.text(0.97, 1.05, f"p-value: {pvalue:.2e}", ha="right", va="center",
                transform=ax.transAxes, fontsize=9)
    ax.set(xlabel=f"Time UTC ({pd.Timestamp(dtime[0]).date()})", ylabel="Light Yield (b3)",
           title=f"Run {run_id}")
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%H:%M"))
    plt.tight_layout(); plt.show()

    print(f"Run {run_id}, {np.nanmedian(ly):.2f}+-{np.nanstd(ly):.2f}, dust={dust:.3e}, flag={run_id in discarded_runs}, ZD={np.mean(tab['mean_zenith_distance']):.2f}deg")
    rw_run.append(run_id)
    rw_time.append(np.mean(tstamp))
    rw_median.append(np.nanmedian(ly))
    rw_std.append(np.nanstd(ly))
    rw_dust.append(dust)
    rw_zd.append(np.mean(tab["mean_zenith_distance"]))
    rw_flag.append(run_id in discarded_runs)
    rw_intensity.append(np.mean(tab["intensity_at_half_peak_rate"]))


In [ ]:
rw_run = np.array(rw_run)
rw_flag = np.array(rw_flag)
rw_median = np.array(rw_median)
rw_std = np.array(rw_std)
rw_time = np.array(rw_time)
rw_dust = np.array(rw_dust)
rw_zd = np.array(rw_zd)
rw_intensity = np.array(rw_intensity)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 3))
x = rw_run.astype(str)
ax.errorbar(x,  rw_median,  yerr=rw_std,  ls="", marker="")
ax.errorbar(x[rw_flag],  rw_median[rw_flag],  yerr=rw_std[rw_flag],  ls="", lw=0, marker=".", ms=15, label="IRREG", color="r", alpha=0.7)
ax.errorbar(x, rw_median, yerr=rw_std, ls="", marker=".", color="C2", label="ZD < 45 deg")

mm = (rw_zd > 45)
ax.errorbar(x[mm], rw_median[mm], yerr=rw_std[mm], ls="", marker=".", label="ZD > 45 deg", color="darkorange")

mm = (rw_dust > 2)
ax.errorbar(x[mm], rw_median[mm], ls="", marker=".", label="Dust > 2 ug/m3", color="k", alpha=1)

ax.axhline(1, ls="--", color="0.3")
ax.tick_params(axis="x", labelrotation=90, labelsize=7)
ax.grid(alpha=0.3); ax.legend(ncols=2, loc=4)
ax.set(ylabel="Light Yield (b3)", ylim=(0.7, 1.15))
plt.tight_layout()
plt.show()

In [ ]:
final_run_sel = rw_run[~rw_flag & (rw_zd > 0)]
final_run_sel

## Step 8 - Linear interpolation: save per-subrun light yield for final run selection


In [ ]:
rows = []
for i, run_id in enumerate(final_run_sel):
    run_id = int(run_id)
    fr = fit_results.get(run_id)
    if fr is None:
        print(f"Run {run_id}: no fit results stored - skip")
        continue

    p0, p1 = fr["p0"], fr["p1"]
    if np.isnan(p0) or np.isnan(p1):
        print(f"Run {run_id}: fit unavailable (p0/p1 NaN) - skip")
        continue

    # Predict light_yield via linear interpolation (the run's linear fit)
    # at every subrun's timestamp
    ly_pred = utils.straight_line(fr["tstamp"], p0, p1)
    for srun, ly in zip(fr["subrun"], ly_pred):
        rows.append((run_id, int(srun), float(ly), rw_intensity[i]))

df_out = pd.DataFrame(rows, columns=["runnumber", "srun", "light_yield", "intensity_at_half_peak_rate"])
df_out = df_out.sort_values(["runnumber", "srun"]).reset_index(drop=True)

Table.from_pandas(df_out).write(FNAME_FINAL_LY3, format="ascii.ecsv", overwrite=True)
print(f"Saved {len(df_out):,} subrun rows from {df_out['runnumber'].nunique():,} runs -> {FNAME_FINAL_LY3}")
df_out.head()
